In [1]:
print("hi")

hi


In [1]:
import os
import re
import json
import pdfplumber
# from pipe_fn import pipe
from pypdf import PdfReader, PdfWriter
import pdfplumber
import os
from transformers import pipeline
from PIL import Image
from output_utils import save_split_output

from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)




COLUMN_HEADERS = [
    "provider",
    "service_code",
    "date_of_service",
    "billed_charges",
    "contracted_charges",
    "disallowed_charges",
    "deductible_charges",
    "copay_coins_amount",
    "remaining_member_expense",
    "amount_paid"
]


SCHEMA = {
    "rows": [
        {
            "service_code": "",
            "date_of_service": "",
            "billed_charges": "",
            "contracted_charges": "",
            "disallowed_charges": "",
            "deductible_charges": "",
            "copay_coins_amount": "",
            "remaining_member_expense": "",
            "amount_paid": ""
        }
    ],

    "claim_total": {
        "amount_paid": ""
    }
}

PROMPT = """ Extract the table data into valid JSON format using the provided schema.

RULES:

IMPORTANT ROW EXTRACTION RULE:

Each service entry in this PDF is visually split into TWO horizontal lines.

FIRST LINE contains:
- service_code
- date_of_service

SECOND LINE contains:
- billed_charges
- contracted_charges
- disallowed_charges
- deductible_charges
- copay_coins_amount
- remaining_member_expense
- amount_paid

These TWO lines together represent ONLY ONE service row.

DO NOT create separate rows for upper line and lower line.

Merge both visual lines into a single JSON row.

If a service_code already appeared in immediately previous row with same date_of_service and same values, DO NOT repeat it.

One service_code must produce only one JSON object.


HEADER EXTRACTION

Extract these fields BEFORE reading the table.

provider =
The text immediately following "Vendor Name:"

Example

Vendor Name: DUC TANG DDS PLLC

Output

"provider":"DUC TANG DDS PLLC"

Never return empty if Vendor Name is visible.

1. service_code:
- Extract ONLY the service code column.
- service_code must strictly start with "D" followed by exactly 4 numbers.
- Example:
  D1110
  D0120
- Do NOT extract descriptions as service_code.
- If invalid or missing, return "".

2. billed_charges:
- Extract ONLY from the "Billed Charges" column.
- Preserve the "$" symbol exactly as shown.
- Example:
  "$149.00"

3. contracted_charges:
- Extract ONLY from the "Contracted Charges" column.
- Preserve "$" symbol.

4. date_of_service:
- Extract ONLY the date near billed/contracted charges.
- Date format must be:
  MM/DD/YYYY
- Example:
  12/18/2025

5. disallowed_charges:
- Extract ONLY from the "Disallowed Charges" column.
- Preserve "$" symbol.

6. deductible_charges:
- Extract ONLY from the "Deductible Charges" column.
- Preserve "$" symbol.

7. copay_coins_amount:
- Extract ONLY from the "Copay/Coins Amount" column.
- Preserve "$" symbol.

8. remaining_member_expense:
- Extract ONLY from the "Remaining Member Expense" column.
- Preserve "$" symbol.

9. amount_paid:
- Extract ONLY from the "Amount Paid" column.
- Preserve "$" symbol.

10. claim_total:
- Claim Total is mentioned at the bottom-right corner of the table.
- Extract ONLY the amount value.
- Preserve "$" symbol.

11. IMPORTANT:
- Do NOT hallucinate values.
- Do NOT merge columns.
- Do NOT shift values between columns.
- If value is missing return "".
- Return ONLY valid JSON.
8. OUTPUT FORMAT

Return ONLY valid JSON in the following structure:

{
    "rows": [
        {
            "provider": {
                "value": "",
                "confidence": 0.0
            },

            "service_code": {
                "value": "",
                "confidence": 0.0
            },

            "date_of_service": {
                "value": "",
                "confidence": 0.0
            },

            "billed_charges": {
                "value": "",
                "confidence": 0.0
            },

            "contracted_charges": {
                "value": "",
                "confidence": 0.0
            },

            "disallowed_charges": {
                "value": "",
                "confidence": 0.0
            },

            "deductible_charges": {
                "value": "",
                "confidence": 0.0
            },

            "copay_coins_amount": {
                "value": "",
                "confidence": 0.0
            },

            "remaining_member_expense": {
                "value": "",
                "confidence": 0.0
            },

            "amount_paid": {
                "value": "",
                "confidence": 0.0
            }
        }
    ],

    "claim_total": {
        "amount_paid": {
            "value": "",
            "confidence": 0.0
        }
    }
}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

"""


def extract_table_from_image(image_path):

    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": PROMPT
                }
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=5000,
        temperature = 0.0,
        repetition_penalty = 1.2
    )

    generated_text = output[0]["generated_text"]

    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    generated_text = generated_text.strip()

    # REMOVE MARKDOWN IF MODEL RETURNS IT
    generated_text = generated_text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    return generated_text


def enforce_schema(parsed_output):

    allowed_columns = COLUMN_HEADERS  

    for table in parsed_output.get("tables", []):

        cleaned_rows = []

        for row in table.get("rows", []):

            cleaned_row = {}

            for col in allowed_columns:
                cleaned_row[col] = row.get(col, "")

            service_code = cleaned_row.get(
                "service_code",
                ""
            ).strip()

            if not re.match(r"^D\d{4}$", service_code):
                continue

            cleaned_rows.append(cleaned_row)
        table["rows"] = cleaned_rows

        # =========================
        # 🔹 CLEAN COLUMN TOTALS (STRICT + ORDERED)
        # =========================
        claim_total = table.get("claim_total", {})
        table["claim_total"] = {"amount_paid": claim_total.get("amount_paid","")}

    return parsed_output

# =========================================================
# MAIN PIPELINE
# =========================================================




def check_claim_denied(page):

    """
    Logic:
    Search ONLY before:
    'Attention Non-contracted Medicare Providers'

    If denied / denial keywords exist before that section,
    return True else False
    """

    full_text = page.extract_text()

    if not full_text:
        return False

    # -----------------------------------------------------
    # LIMIT SEARCH AREA
    # -----------------------------------------------------

    stop_keyword = "For Claim Submissions and ReSubmissions:"

    if stop_keyword in full_text:

        full_text = full_text.split(stop_keyword)[0]

    searchable_text = full_text.lower()

    # -----------------------------------------------------
    # DENIAL KEYWORDS
    # -----------------------------------------------------

    denial_keywords = [

        "denied",
        "denial",
    ]

    # -----------------------------------------------------
    # SEARCH
    # -----------------------------------------------------

    for keyword in denial_keywords:

        if keyword in searchable_text:

            print(f"❌ Claim Denied Keyword Found: {keyword}")

            return "denied"

    return "not denied"

def rotate_and_crop_pdf(
    pdf_path,
    output_dir="EOB_OUTPUT/Blue_shield", company_name = "Blue shield"
):
    
    pdf_file = os.path.basename(pdf_path)
    pdf_name = pdf_file.split("_")[-1].split(".")[0]
    pdf_full_name = os.path.basename(pdf_path)


    # =====================================
    # CREATE OUTPUT FOLDER
    # =====================================

    pdf_dir = os.path.basename(pdf_path).split(".")[0].split("_")[-1]

    output_dir = os.path.join(output_dir, pdf_dir)

    os.makedirs(output_dir, exist_ok=True)

    all_patients = []
    confidence_results = [] 

    # =====================================
    # ROTATE PDF
    # =====================================

    rotated_pdf_path = os.path.join(
        output_dir,
        f"rotated_{pdf_dir}.pdf"
    )

    reader = PdfReader(pdf_path)

    writer = PdfWriter()

    for page_num, page in enumerate(reader.pages, start=1):

        is_denied = check_claim_denied(page)

        print(f"Denied Status : {is_denied}")
        print(f"\n🚀 Processing Page {page_num}")

        if page_num != 1:

            page.rotate(90)

        else:

            print("⏭️ Keeping first page normal")
        writer.add_page(page)

    with open(rotated_pdf_path, "wb") as f:

        writer.write(f)

    print(f"\n✅ Rotated PDF Saved: {rotated_pdf_path}")

    # =====================================
    # OPEN ROTATED PDF
    # =====================================

    

    with pdfplumber.open(rotated_pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            print(f"\n🧾 Cropping Page {page_num}")

            start_positions = []
            end_positions = []

            start_hits = page.search("Claim #")

            end_hits = page.search("Federal Employee Program")

            if not start_hits:

                start_hits = page.search("Claim")

            if not end_hits:

                end_hits = page.search("Claim Total:")

            for hit in start_hits:

                start_positions.append(hit["top"] - 10)

            for hit in end_hits:

                end_positions.append(hit["bottom"] + 20)

            table_count = min(
                len(start_positions),
                len(end_positions)
            )

            if table_count == 0:

                print("❌ No tables found")

                continue

            # =====================================
            # CROP TABLES
            # =====================================

            for idx in range(table_count):

                print(f"\n🧾 Processing Table {idx + 1}")

                start_y = start_positions[idx]

                end_y = end_positions[idx]

                bbox = (
                    0,
                    start_y,
                    page.width,
                    end_y
                )

                cropped_page = page.crop(bbox)

                image_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )

                cropped_page.to_image(resolution=300).save(image_path)

                print(f"✅ Saved: {image_path}")

                # =====================================
                # MODEL EXTRACTION
                # =====================================

                try:

                    llm_output = extract_table_from_image(image_path)

                    print("\n================ RAW MODEL OUTPUT ================\n")

                    print(llm_output)

                    print("\n==================================================\n")

                    parsed_output = json.loads(llm_output)

                    model_confidence = calculate_model_confidence(parsed_output)   # ADD
                    parsed_output = _unwrap_vlm_output(parsed_output) 

                    parsed_output = {
                        "tables": [parsed_output]
                    }

                    parsed_output = enforce_schema(parsed_output)

                    for table in parsed_output.get("tables", []):        # ADD
                        table["_model_confidence"] = model_confidence 

                    # =====================================
                    # STRUCTURED OUTPUT
                    # =====================================

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        expected_rows = count_service_rows(page, start_y, end_y)

                        row_count_ok = validate_service_row_count(
                            page,
                            start_y,
                            end_y,
                            table,
                            t_idx
                        )

                        is_valid, log, errors, total_fields  = validate_eob_table(table, t_idx)
                        if not row_count_ok:
                            is_valid = False
                            errors = errors + [{"type": "row_count_mismatch"}]

                        structured_table = {

                            "EOB_ID": pdf_name,
                            "patient_name": extract_patient_name(page),
                            "claim_status": is_denied,
                            "rows": table.get("rows", []),

                            "claim_total": table.get("claim_total", {}),
                            "validation": {"status": is_valid, "errors": errors}, 
                            "_expected_rows": expected_rows,                          # ADD
                            "_total_fields": total_fields,                             # ADD
                            "_model_confidence": table.get("_model_confidence", 0.0),
                        }

                        print(f"✅ Completed Table {idx + 1} on Page {page_num}")

                        # results.append(structured_table)
                    date_of_service = ""
                    provider = ""

                    if structured_table.get("rows"):
                        date_of_service = structured_table["rows"][0].get("date_of_service", "")
                        provider = structured_table["rows"][0].get("provider", "")

                    for row in structured_table.get("rows", []):

                        for col in ["service_description", "date_of_service", "provider"]:
                            row.pop(col, None)

                    # structured_table["totals"].pop("service_description", None)

                    patient_data = {
                        # "EOB_id": structured_table.get("EOB_ID", ""),
                        "patient_name": structured_table.get("patient_name", ""),
                        "provider": provider,                    
                        "date_of_service": date_of_service,
                        "services": structured_table.get("rows", []),
                        "totals": structured_table.get("claim_total", {}),
                        "validation": structured_table.get("validation"),
                        "_expected_rows": structured_table.get("_expected_rows", 0),               # ADD
                        "_total_fields": structured_table.get("_total_fields", 0),                  # ADD
                        "_model_confidence": structured_table.get("_model_confidence", 0.0),      
                    }

                    all_patients.append(patient_data)
                    confidence_results.append(patient_data)   # ADD
                    

                except Exception as e:

                    print(f"❌ Extraction Failed: {str(e)}")

                    all_patients.append({

                        "page": page_num,

                        "table": idx + 1,

                        "status": "failed",

                        "image_path": image_path,

                        "error": str(e)
                    })
    confidence_score = calculate_eob_confidence(confidence_results)   # ADD

    for patient in all_patients:                                      # ADD cleanup
        patient.pop("_expected_rows", None)
        patient.pop("_total_fields", None)
        patient.pop("_model_confidence", None)
    
    final_output=[

        
        {
        "eob_id": pdf_name,
        "file_name":pdf_full_name,
        "claim_status": is_denied,
        "payor": "Blue Cross Blue Shield of North Carolina",
        "confidence_score": confidence_score,  
        "patients": all_patients
    }
]       

    success_path, failed_path = save_split_output(
                    final_output,
                    company_name=company_name,
                    pdf_name=pdf_name,
                    pdf_path=pdf_path,
                    cropped_dir=output_dir,
                )
            
    print(f"\n📁 Cropped images : {output_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final_output




    

#------------------- PARSE AMOUNT --------------------#

def parse_amount(val):
    if val in ["", None]:
        return 0.0
    return float(str(val).replace("$", "").replace(",", "").strip())

#------------------- VALIDATE TOTAL AMOUNT --------------------#


def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()
    row_positions = []

    for w in words:
        text = w["text"].strip()

        match = re.search(r"\bD\d{4}\b", text)

        if match:
            y = float(w["top"])

            if region_top <= y <= region_bottom:
                row_positions.append(y)

    row_positions.sort()

    grouped_rows = []
    threshold = 3

    for y in row_positions:
        if not grouped_rows:
            grouped_rows.append(y)
        else:
            if abs(y - grouped_rows[-1]) > threshold:
                grouped_rows.append(y)

    return len(grouped_rows)

def extract_patient_name(page):

    text = page.extract_text()

    if not text:
        return ""

    match = re.search(
        r"Patient\s+Name:\s*(.+)",
        text
    )

    if match:

        name = match.group(1).strip()

        name = name.split("\n")[0].strip()

        return name

    return ""

def validate_eob_table(table: dict, table_index: int):

    rows = table.get("rows", [])
    claim_total = table.get("claim_total", {})
    table["claim_total"] = {"amount_paid": claim_total.get("amount_paid","")}

    if not rows:
        return False, "", [], total_fields 

    computed_totals = {
    "amount_paid": round(sum(parse_amount(r.get("amount_paid", "")) for r in rows), 2)
}
    total_fields = len(computed_totals)
    
    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [Table {table_index}]")
    print("-" * 75)

    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(claim_total.get(field, "")), 2)

        if computed_value == extracted_value:
            icon = "✅"
            status = "match"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    if has_error:
        print(f"❌ [Table {table_index}] Validation FAILED\n")
        return False, result_validation, errors, total_fields    
    else:
        print(f"✅ [Table {table_index}] Validation PASSED\n")
        return True, result_validation, [], total_fields    

def validate_service_row_count(page, start_y, end_y, table, table_index):

    detected_count = count_service_rows(page, start_y, end_y)

    rows = table.get("rows", [])
    extracted_count = len([
        r for r in rows
        if r.get("service_code") not in ["", None]
    ])

    print(f"\n📊 Row Count Validation [Table {table_index}]")
    print("-" * 70)

    if detected_count == extracted_count:
        icon = "✅"
        status = "match"
    else:
        icon = "❌"
        status = "MISMATCH"

    print(f"{icon} row_count detected={detected_count:<5} | extracted={extracted_count:<5} {status}")
    print("-" * 70)


    return detected_count == extracted_count





W0901 18:05:50.690000 3380944 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 18:05:50.706000 3380944 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
rotate_and_crop_pdf(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Blue_shield/PDF's/Pmt_EOP_853275678.pdf")

Denied Status : not denied

🚀 Processing Page 1
⏭️ Keeping first page normal
Denied Status : not denied

🚀 Processing Page 2
Denied Status : not denied

🚀 Processing Page 3
Denied Status : not denied

🚀 Processing Page 4

✅ Rotated PDF Saved: EOB_OUTPUT/Blue_shield/853275678/rotated_853275678.pdf

🧾 Cropping Page 1
❌ No tables found

🧾 Cropping Page 2

🧾 Processing Table 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Saved: EOB_OUTPUT/Blue_shield/853275678/page_2_table_1.png


[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`



================ RAW MODEL OUTPUT ================

{
    "rows": [
        {
            "provider": "DUC TANG DDS PLLC",
            "service_code": "D0120",
            "date_of_service": "06/26/2025",
            "billed_charges": "$89.00",
            "contracted_charges": "$53.00",
            "disallowed_charges": "$36.00",
            "deductible_charges": "$0.00",
            "copay_coins_amount": "$35.00",
            "remaining_member_expense": "$35.00",
            "amount_paid": "$18.00"
        },
        {
            "provider": "DUC TANG DDS PLLC",
            "service_code": "D1110",
            "date_of_service": "06/26/2025",
            "billed_charges": "$149.00",
            "contracted_charges": "$95.00",
            "disallowed_charges": "$54.00",
            "deductible_charges": "$0.00",
            "copay_coins_amount": "$0.00",
            "remaining_member_expense": "$0.00",
            "amount_paid": "$95.00"
        }
    ],
    "claim_total": {
       

[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



================ RAW MODEL OUTPUT ================

{
    "rows": [
        {
            "provider": "DUC TANG DDS PLLC",
            "service_code": "D1110",
            "date_of_service": "09/23/2025",
            "billed_charges": "$149.00",
            "contracted_charges": "$0.00",
            "disallowed_charges": "$149.00",
            "deductible_charges": "$0.00",
            "copay_coins_amount": "$0.00",
            "remaining_member_expense": "$149.00",
            "amount_paid": "$0.00"
        }
    ],
    "claim_total": {
        "amount_paid": {
            "value": "$0.00",
            "confidence": 1.0
        }
    }
}



📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count detected=1     | extracted=1     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ amount_paid      

[{'eob_id': '853275678',
  'file_name': 'Pmt_EOP_853275678.pdf',
  'claim_status': 'not denied',
  'payor': 'Blue Cross Blue Shield of North Carolina',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'DELONG, CHRIS A',
    'provider': 'DUC TANG DDS PLLC',
    'date_of_service': '06/26/2025',
    'services': [{'service_code': 'D0120',
      'billed_charges': '$89.00',
      'contracted_charges': '$53.00',
      'disallowed_charges': '$36.00',
      'deductible_charges': '$0.00',
      'copay_coins_amount': '$35.00',
      'remaining_member_expense': '$35.00',
      'amount_paid': '$18.00'},
     {'service_code': 'D1110',
      'billed_charges': '$149.00',
      'contracted_charges': '$95.00',
      'disallowed_charges': '$54.00',
      'deductible_charges': '$0.00',
      'copay_coins_amount': '$0.00',
      'remaining_member_expense': '$0.00',
      'amount_paid': '$95.00'}],
    'totals': {'amount_paid': '$113.00'},
    'validation': {'status': True, 'errors': []}},
   {'

In [2]:
rotate_and_crop_pdf(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Blue_shield/PDF's/Pmt_EOP_839328115.pdf")

Denied Status : not denied

🚀 Processing Page 1
⏭️ Keeping first page normal
Denied Status : not denied

🚀 Processing Page 2

✅ Rotated PDF Saved: EOB_OUTPUT/Blue_shield/839328115/rotated_839328115.pdf

🧾 Cropping Page 1
❌ No tables found

🧾 Cropping Page 2

🧾 Processing Table 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Saved: EOB_OUTPUT/Blue_shield/839328115/page_2_table_1.png

================ RAW MODEL OUTPUT ================

{
    "rows": [
        {
            "provider": "DUC TANG DDS PLLC",
            "service_code": "D1110",
            "date_of_service": "12/18/2025",
            "billed_charges": "$149.00",
            "contracted_charges": "$95.00",
            "disallowed_charges": "$54.00",
            "deductible_charges": "$0.00",
            "copay_coins_amount": "$0.00",
            "remaining_member_expense": "$0.00",
            "amount_paid": "$95.00"
        },
        {
            "provider": "DUC TANG DDS PLLC",
            "service_code": "D0120",
            "date_of_service": "12/18/2025",
            "billed_charges": "$89.00",
            "contracted_charges": "$53.00",
            "disallowed_charges": "$36.00",
            "deductible_charges": "$0.00",
            "copay_coins_amount": "$35.00",
            "remaining_member_expense": "$35.00",
            "amount_

[{'eob_id': '839328115',
  'file_name': 'Pmt_EOP_839328115.pdf',
  'claim_status': 'not denied',
  'payor': 'Blue Cross Blue Shield of North Carolina',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'DELONG, CHRIS A',
    'provider': 'DUC TANG DDS PLLC',
    'date_of_service': '12/18/2025',
    'services': [{'service_code': 'D1110',
      'billed_charges': '$149.00',
      'contracted_charges': '$95.00',
      'disallowed_charges': '$54.00',
      'deductible_charges': '$0.00',
      'copay_coins_amount': '$0.00',
      'remaining_member_expense': '$0.00',
      'amount_paid': '$95.00'},
     {'service_code': 'D0120',
      'billed_charges': '$89.00',
      'contracted_charges': '$53.00',
      'disallowed_charges': '$36.00',
      'deductible_charges': '$0.00',
      'copay_coins_amount': '$35.00',
      'remaining_member_expense': '$35.00',
      'amount_paid': '$18.00'}],
    'totals': {'amount_paid': '$113.00'},
    'validation': {'status': True, 'errors': []}}]}]